In [45]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno


# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
# from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib
from sklearn import tree 
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier 

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay

# Custom Functions
from credit_risk_modeling import model_eval, scoring
import importlib
# importlib.reload(model_eval)
importlib.reload(scoring)

<module 'credit_risk_modeling.scoring' from 'C:\\Users\\billy\\OneDrive\\Documents\\Finance_Projects\\credit_risk_modeling\\src\\credit_risk_modeling\\scoring.py'>

In [46]:
# Load RAW (12-feature) credit risk dataset - not preprocessed
df_interim = pd.read_csv("../data/interim/credit_risk_dataset_prepped.csv")
print(f"Raw interim dataset shape: {df_interim.shape}")
print(f"Columns: {df_interim.columns.tolist()}")
print(f"\nFirst row:\n{df_interim.iloc[0]}")

Raw interim dataset shape: (32574, 12)
Columns: ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']

First row:
person_age                           21
person_income                      9600
person_home_ownership               OWN
person_emp_length                   5.0
loan_intent                   EDUCATION
loan_grade                            B
loan_amnt                          1000
loan_int_rate                     11.14
loan_status                       False
loan_percent_income                 0.1
cb_person_default_on_file         False
cb_person_cred_hist_length            2
Name: 0, dtype: object


In [47]:
model, preprocessor = scoring.load_model_and_preprocessor()

2026-02-09 10:04:54.163 | INFO     | credit_risk_modeling.scoring:load_model_and_preprocessor:15 - ✓ Model and preprocessor loaded successfully


In [48]:
# Also load the raw features to get loan_intent (for decision rules later)
# df_raw = pd.read_csv("../data/interim/credit_risk_dataset_prepped.csv")
# y_test = pd.read_csv("../data/interim/y_test.csv")

# print(f"Raw dataset shape: {df_raw.shape}")
# print(f"Test labels shape: {y_test.shape}")

In [49]:
# Extract a single applicant from RAW data (12 features)
applicant_idx = 0
applicant_interim = df_interim.iloc[applicant_idx].drop('loan_status').to_dict()

print(f"✓ Sample applicant (raw, {len(applicant_interim)} features):")
print(f"  Index: {applicant_idx}")
print(f"  Features:")
for key, val in applicant_interim.items():
    print(f"    {key}: {val}")

✓ Sample applicant (raw, 11 features):
  Index: 0
  Features:
    person_age: 21
    person_income: 9600
    person_home_ownership: OWN
    person_emp_length: 5.0
    loan_intent: EDUCATION
    loan_grade: B
    loan_amnt: 1000
    loan_int_rate: 11.14
    loan_percent_income: 0.1
    cb_person_default_on_file: False
    cb_person_cred_hist_length: 2


In [55]:
# print("Preprocessor type:", type(preprocessor))
# print("\nPreprocessor steps:")
# for name, step in preprocessor.named_steps.items():
#     print(f"  {name}: {type(step).__name__}")
#     if hasattr(step, 'transformers_'):
#         for trans_name, transformer, cols in step.transformers_:
#             print(f"    - {trans_name}: {cols if cols != 'passthrough' else 'passthrough'}")

# print("\nInput X columns:", X.columns.tolist())
# print("Input X shape:", X.shape)
# print("\nX.dtypes:")
# print(X.dtypes)

In [56]:
# preprocessor.transform(X)

In [58]:
applicant_interim

{'person_age': 21,
 'person_income': 9600,
 'person_home_ownership': 'OWN',
 'person_emp_length': 5.0,
 'loan_intent': 'EDUCATION',
 'loan_grade': 'B',
 'loan_amnt': 1000,
 'loan_int_rate': 11.14,
 'loan_percent_income': 0.1,
 'cb_person_default_on_file': False,
 'cb_person_cred_hist_length': 2}

In [ ]:
# First, let's see what preprocessor.transform actually returns
X_test = pd.DataFrame([applicant_interim])
if 'cb_person_default_on_file' in X_test.columns and X_test['cb_person_default_on_file'].dtype == 'bool':
    X_test['cb_person_default_on_file'] = X_test['cb_person_default_on_file'].astype('object')

X_transformed = preprocessor.transform(X_test)

print("X_transformed type:", type(X_transformed))
print("X_transformed shape:", X_transformed.shape)
if isinstance(X_transformed, pd.DataFrame):
    print("\nX_transformed dtypes:")
    print(X_transformed.dtypes)
    print("\nX_transformed columns:")
    print(X_transformed.columns.tolist())
else:
    print("X_transformed is a numpy array")
    print("X_transformed dtype:", X_transformed.dtype)


2026-02-09 10:04:58.535 | WARNING  | credit_risk_modeling.scoring:calculate_pd:45 - Preprocessing failed, attempting direct prediction: "None of [Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',\n       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length'],\n      dtype='object')] are in the [columns]"


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: person_home_ownership: object, loan_intent: object, loan_grade: object, cb_person_default_on_file: object

In [59]:
scoring.calculate_pd(
    features = applicant_interim,
    model = model,
    preprocessor = preprocessor
)

2026-02-09 10:16:30.321 | WARNING  | credit_risk_modeling.scoring:calculate_pd:45 - Preprocessing failed, attempting direct prediction: "None of [Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',\n       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length'],\n      dtype='object')] are in the [columns]"


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: person_home_ownership: object, loan_intent: object, loan_grade: object, cb_person_default_on_file: object

In [ ]:
X = pd.DataFrame([applicant_raw])

In [ ]:
X

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,0.59,Y,3


In [ ]:
preprocessor.transform(X)

ValueError: Input columns ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6'] have low variation for method 'iqr'. Try other capping methods or drop these columns.

In [ ]:
try:
    # Calculate PD using scoring module  (passes through preprocessor + model)
    pd_value = scoring.calculate_pd(applicant_raw, model, preprocessor)
    
    print(f"\n✓ SUCCESS! PD calculated from {len(applicant_raw)} raw features")
    print(f"  Probability of Default: {pd_value:.4f}")
    print(f"  As percentage: {pd_value*100:.2f}%")
    print("="*60)
    
except Exception as e:
    print(f"\n✗ ERROR: {e}")
    print(f"\nDebug info:")
    print(f"  Applicant features (12): {list(applicant_raw.keys())}")
    print(f"  Model: {type(model)}")
    print(f"  Preprocessor: {type(preprocessor)}")
    print("="*60)

2026-02-08 13:03:42.339 | WARNING  | credit_risk_modeling.scoring:calculate_pd:43 - Preprocessing failed, attempting direct prediction: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

✗ ERROR: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: person_home_ownership: object, loan_intent: object, loan_grade: object, cb_person_default_on_file: object

Debug info:
  Applicant features (12): ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']
  Model: <class 'sklearn.model_selection._classification_threshold.FixedThresholdClassifier'>
  Preprocessor: <class 'sklearn.pipeline.Pipeline'>


In [ ]:
#  🔹 STEP 1 TEST: Test scoring calculations (bypassing model for now)
# We'll test the business logic without the model/preprocessor compatibility issue

# Manually set test values that represent a sample applicant's scoring
pd_test = 0.18  # Model would predict this
lgd_test = 1.0  # Phase 2: 100% loss
ead_test = loan_amnt  # Loan amount

print("="*60)
print("STEP 1: Test Scoring Calculations")
print("="*60)
print(f"\nTest Inputs:")
print(f"  Probability of Default (PD): {pd_test:.4f} ({pd_test*100:.2f}%)")
print(f"  Loss Given Default (LGD):    {lgd_test:.2f} ({lgd_test*100:.0f}%)")
print(f"  Exposure at Default (EAD):   ${ead_test:,.2f}")

# Test calculate_expected_loss
el_test = scoring.calculate_expected_loss(pd_test, lgd_test, ead_test)
print(f"\n✓ Expected Loss = PD × LGD × EAD")
print(f"  EL = {pd_test:.4f} × {lgd_test} × ${ead_test:,.0f}")
print(f"  EL = ${el_test:,.2f}")

# Test normalize_to_risk_score
risk_score, risk_tier = scoring.normalize_to_risk_score(pd_test)
print(f"\n✓ Risk Scoring")
print(f"  Risk Score: {risk_score}/100")
print(f"  Risk Tier: {risk_tier}")

# Test calculate_confidence
confidence = scoring.calculate_confidence(pd_test)
print(f"\n✓ Model Confidence: {confidence:.2f} ({confidence*100:.0f}%)")

print("\n" + "="*60)
print("✓ STEP 1 COMPLETE: All scoring calculations working!")
print("="*60)

2026-02-08 11:56:20.645 | WARNING  | credit_risk_modeling.scoring:calculate_pd:43 - Preprocessing failed, attempting direct prediction: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


LightGBMError: The number of features in data (19) is not the same as it was in training data (25).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.

In [ ]:
# Manually compute all scoring components (mirroring what score_applicant() does)
import numpy as np

# 1. Calculate PD
pd_value = pd_value  # Already calculated above

# 2. Calculate EAD (loan amount)
ead_value = loan_amnt

# 3. Calculate LGD (Phase 2 = 100%)
lgd_value = 1.0

# 4. Calculate Expected Loss
el_value = pd_value * lgd_value * ead_value

# 5. Calculate risk score and tier
risk_score = int(np.clip(pd_value * 100, 0, 100))
if risk_score < 33:
    risk_tier = 'LOW'
elif risk_score < 67:
    risk_tier = 'MEDIUM'
else:
    risk_tier = 'HIGH'

# 6. Calculate confidence
confidence = 1 - abs(pd_value - 0.5) * 2
confidence = np.clip(confidence, 0, 1)

# Build the scoring dict
full_score = {
    'pd': pd_value,
    'lgd': lgd_value,
    'ead': ead_value,
    'expected_loss': el_value,
    'risk_score': risk_score,
    'risk_tier': risk_tier,
    'confidence': confidence
}

print("\n" + "="*60)
print("FULL APPLICANT SCORE (Phase 2)")
print("="*60)
print(f"PD (Probability of Default):  {full_score['pd']:.4f} ({full_score['pd']*100:.2f}%)")
print(f"LGD (Loss Given Default):     {full_score['lgd']:.2f} ({full_score['lgd']*100:.0f}%)")
print(f"EAD (Exposure at Default):    ${full_score['ead']:,.2f}")
print(f"Expected Loss:                ${full_score['expected_loss']:,.2f}")
print(f"Risk Score:                   {full_score['risk_score']}/100")
print(f"Risk Tier:                    {full_score['risk_tier']}")
print(f"Confidence:                   {full_score['confidence']:.2f} ({full_score['confidence']*100:.0f}%)")
print("="*60)

In [ ]:
# Test the decision engine with the computed score
from credit_risk_modeling.decision_rules import ApprovalRuleEngine

engine = ApprovalRuleEngine()
decision = engine.decide(
    pd=full_score['pd'],
    lgd=full_score['lgd'],
    ead=full_score['ead'],
    expected_loss=full_score['expected_loss'],
    loan_intent=loan_intent
)

print("\n" + "="*60)
print("APPROVAL DECISION")
print("="*60)
print(f"Decision:     {decision['decision']}")
print(f"EL % of EAD:  {decision['el_pct']*100:.2f}%")
print(f"\nReason:\n{decision['reason']}")
print("="*60)